In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from config import CONFIG

engine = create_engine(CONFIG["db_url"])
print("Connected")

Connected


In [2]:
df = pd.read_sql(
    f'SELECT * FROM "{CONFIG["schema"]}"."{CONFIG["clean_table"]}"',
    engine
)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 9,031 rows x 17 columns


## All Derived Features

In [3]:
import numpy as np
import pandas as pd

def engineer_features_tb(df):
    df_out = df.copy()

    # --- Feature 1: Confidence interval width ---
    df_out["ci_width"] = (df_out["hi"] - df_out["lo"]).round(2)

    # --- Feature 2: Relative CI width (%) ---
    df_out["ci_relative_width"] = (
        (df_out["hi"] - df_out["lo"])
        / df_out["best"].replace(0, np.nan)
        * 100
    ).round(2)

    # --- Feature 3: WHO region ---
    iso3_to_region = {}
    for region, countries in CONFIG.get("who_regions", {}).items():
        for iso3 in countries:
            iso3_to_region[iso3] = region
    df_out["who_region"] = (
        df_out["iso3"]
        .map(iso3_to_region)
        .fillna("Other/Unknown")
    )

    # --- Feature 4: World Bank income group ---
    iso3_to_income = {}
    for group, countries in CONFIG.get("income_groups", {}).items():
        for iso3 in countries:
            iso3_to_income[iso3] = group
    df_out["income_group"] = (
        df_out["iso3"]
        .map(iso3_to_income)
        .fillna("Unknown")
    )

    # --- Feature 5: WHO high-burden country flag ---
    df_out["high_burden_flag"] = (
        df_out["iso3"].isin(CONFIG.get("who_high_burden_30", []))
    ).astype(int)

    # --- Feature 6: East Africa flag ---
    df_out["east_africa_flag"] = (
        df_out["iso3"].isin(CONFIG.get("east_africa", []))
    ).astype(int)

    # --- Summary ---
    new_features = [
        "ci_width", "ci_relative_width", "who_region",
        "income_group", "high_burden_flag", "east_africa_flag"
    ]
    
    print("--- Engineered Features Summary ---")
    for feat in new_features:
        if df_out[feat].dtype in ["float64", "int64"]:
            print(f"{feat:<25}: "
                  f"min={df_out[feat].min():.2f} "
                  f"max={df_out[feat].max():.2f} "
                  f"mean={df_out[feat].mean():.2f}")
        else:
            print(f"{feat:<25}: "
                  f"{df_out[feat].nunique()} unique values")

    return df_out

# Run function
df = engineer_features_tb(df)

--- Engineered Features Summary ---
ci_width                 : min=0.00 max=830000.00 mean=10228.99
ci_relative_width        : min=0.00 max=800.00 mean=177.55
who_region               : 7 unique values
income_group             : 5 unique values
high_burden_flag         : min=0.00 max=1.00 mean=0.16
east_africa_flag         : min=0.00 max=1.00 mean=0.05


## Feature Engineering & Metadata Augmentation

Given that the dataset represents a single cross-sectional snapshot (**2024**), time-series engineered features (e.g., Year-over-Year changes, End TB strategy period bins, 2015 baseline trajectory) were excluded. Instead, feature engineering focused on domain-specific metadata enrichment and surveillance uncertainty modeling:

1. **Uncertainty Metrics:**
   * **`ci_width`**: Absolute margin of uncertainty ($hi - lo$).
   * **`ci_relative_width`**: Relative margin of error expressed as a percentage of the point estimate ($\frac{hi - lo}{best} \times 100$), serving as a proxy for surveillance data precision.

2. **Geographic & Socioeconomic Metadata Mapping:**
   * **`who_region`**: Mapped standard WHO regions (AFRO, AMRO, SEARO, EURO, EMRO, WPRO).
   * **`income_group`**: World Bank national income classifications (High, Upper-Middle, Lower-Middle, Low).
   * **`high_burden_flag`**: Binary indicator ($1/0$) identifying top 30 WHO High TB Burden Nations.
   * **`east_africa_flag`**: Binary regional focus flag ($1/0$).

In [4]:
import pandas as pd
import numpy as np

def validate_pre_export_tb(df):
    """
    Performs critical pre-export sanity checks on the engineered TB dataset.
    Fails fast if structural, numerical, or logic violations exist.
    """
    print("=== Running Pre-Export Validation Audit ===\n")
    validation_passed = True
    
    # 1. Total Row & Column Integrity
    print(f"Total Rows    : {len(df):,}")
    print(f"Total Columns : {len(df.columns)}")
    if len(df) != 9031:
        print(f"❌ CRITICAL: Row count mismatch! Expected 9,031, got {len(df)}")
        validation_passed = False
    else:
        print("✅ Row count integrity verified (9,031 rows).")

    # 2. Check for unexpected NaNs in core or engineered columns
    # Note: ci_relative_width may contain NaNs if best == 0, which is handled
    expected_nan_cols = ["ci_relative_width"] 
    unexpected_nans = df.drop(columns=expected_nan_cols).isna().sum()
    unexpected_nans_found = unexpected_nans[unexpected_nans > 0]
    
    if not unexpected_nans_found.empty:
        print(f"\n❌ CRITICAL: Found unexpected NaNs in columns:")
        print(unexpected_nans_found)
        validation_passed = False
    else:
        print("✅ Zero unexpected NaN values across all features.")

    # 3. Validate Engineered Feature Ranges & Logic
    print("\n--- Engineered Feature Logic Checks ---")
    
    # Check ci_width non-negativity
    invalid_ci_widths = (df["ci_width"] < 0).sum()
    if invalid_ci_widths > 0:
        print(f"❌ CRITICAL: Found {invalid_ci_widths} negative ci_width values!")
        validation_passed = False
    else:
        print("✅ ci_width non-negativity verified.")
        
    # Check Binary Flags (must strictly be 0 or 1)
    binary_flags = ["high_burden_flag", "east_africa_flag"]
    for flag in binary_flags:
        invalid_values = set(df[flag].unique()) - {0, 1}
        if invalid_values:
            print(f"❌ CRITICAL: {flag} contains non-binary values: {invalid_values}")
            validation_passed = False
        else:
            print(f"✅ Flag '{flag}' properly encoded as binary (0/1).")

    # 4. Column Schema & Type Verification
    expected_columns = [
        "country", "iso2", "iso3", "iso_numeric", "year", "measure", "unit",
        "age_group", "sex", "risk_factor", "best", "lo", "hi",
        "ci_width", "ci_relative_width", "who_region", "income_group",
        "high_burden_flag", "east_africa_flag"
    ]
    missing_cols = set(expected_columns) - set(df.columns)
    if missing_cols:
        print(f"\n❌ CRITICAL: Missing expected columns: {missing_cols}")
        validation_passed = False
    else:
        print("✅ All expected schema columns present.")

    # Final Verdict
    print("\n===========================================")
    if validation_passed:
        print("🚀 PRE-EXPORT AUDIT PASSED: Dataset is ready for PostgreSQL export!")
    else:
        print("⚠️ PRE-EXPORT AUDIT FAILED: Fix the errors listed above before exporting.")
    print("===========================================")

# Run validation on engineered DataFrame
validate_pre_export_tb(df)

=== Running Pre-Export Validation Audit ===

Total Rows    : 9,031
Total Columns : 23
✅ Row count integrity verified (9,031 rows).
✅ Zero unexpected NaN values across all features.

--- Engineered Feature Logic Checks ---
✅ ci_width non-negativity verified.
✅ Flag 'high_burden_flag' properly encoded as binary (0/1).
✅ Flag 'east_africa_flag' properly encoded as binary (0/1).
✅ All expected schema columns present.

🚀 PRE-EXPORT AUDIT PASSED: Dataset is ready for PostgreSQL export!


## Pre-Export Pipeline Audit & Validation

Following a complete kernel reset and clean pipeline execution, the dataset successfully passed all pre-export automated quality gates without warnings or errors.

### Final Validation Audit Summary

| Validation Check | Target Threshold / Constraint | Result Status |
| :--- | :--- | :---: |
| **Row Integrity** | Exactly 9,031 observations | **Pass (9,031/9,031)** |
| **Schema Completeness** | 23 expected raw + engineered features | **Pass (23 Columns)** |
| **Missingness Check** | 0 unexpected `NaN` values across all fields | **Pass (0 NaNs)** |
| **Interval Non-Negativity** | $ci\_width \ge 0$ | **Pass** |
| **Binary Encoding Integrity** | $high\_burden\_flag, east\_africa\_flag \in \{0, 1\}$ | **Pass** |

---

### Pipeline Milestone
The clean, feature-engineered Tuberculosis 2024 dataset has been validated as production-ready and is now primed for direct PostgreSQL database persistence and Exploratory Data Analysis (EDA).

In [7]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, types

# 1. Load environment variables from .env file
load_dotenv()

# 2. Fetch credentials safely from environment
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")  # Fallback to localhost if not set
DB_PORT = os.getenv("DB_PORT", "5432")       # Fallback to 5432 if not set
DB_NAME = os.getenv("DB_NAME")

# Ensure critical environment variables exist before connecting
if not all([DB_USER, DB_PASSWORD, DB_NAME]):
    raise ValueError("❌ Missing required database credentials in .env file! Check DB_USER, DB_PASSWORD, and DB_NAME.")

# 3. Create SQLAlchemy Database Engine
connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

# 4. Explicit SQL Data Type Mapping
dtype_mapping = {
    "country": types.VARCHAR(255),
    "iso2": types.VARCHAR(10),
    "iso3": types.VARCHAR(10),
    "iso_numeric": types.VARCHAR(10),
    "year": types.INTEGER(),
    "measure": types.VARCHAR(50),
    "unit": types.VARCHAR(50),
    "age_group": types.VARCHAR(50),
    "sex": types.VARCHAR(20),
    "risk_factor": types.VARCHAR(50),
    "best": types.FLOAT(precision=53),
    "lo": types.FLOAT(precision=53),
    "hi": types.FLOAT(precision=53),
    "ci_width": types.FLOAT(precision=53),
    "ci_relative_width": types.FLOAT(precision=53),
    "who_region": types.VARCHAR(100),
    "income_group": types.VARCHAR(100),
    "high_burden_flag": types.SMALLINT(),
    "east_africa_flag": types.SMALLINT()
}

# 5. Export to PostgreSQL
table_name = "tb_burden_2024_engineered"

try:
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",  # Drops existing table and recreates schema
        index=False,           # Excludes pandas index
        chunksize=1000,        # Batch processing
        dtype=dtype_mapping
    )
    print(f"✅ Successfully exported {len(df):,} rows to PostgreSQL table '{table_name}'.")

except Exception as e:
    print(f"❌ PostgreSQL Export Failed: {e}")

# 6. Local Backup Export (CSV)
df.to_csv("tb_burden_2024_engineered.csv", index=False)
# df.to_parquet("tb_burden_2024_engineered.parquet", index=False)  # Commented out

print("✅ Local CSV backup saved successfully.")

✅ Successfully exported 9,031 rows to PostgreSQL table 'tb_burden_2024_engineered'.
✅ Local CSV backup saved successfully.


## Data Engineering Pipeline Summary & Persistence

The entire data preparation phase has successfully concluded. The 2024 WHO Tuberculosis dataset has been cleansed, validated, and enriched with domain-specific surveillance features.

### Final Pipeline Accomplishments
* **Dataset Size:** 9,031 complete records $\times$ 23 standardized & engineered attributes.
* **Database Target:** Persisted to PostgreSQL relational table `tb_burden_2024_engineered` via `sqlalchemy` batch execution (`chunksize=1000`).
* **Artifact Backups:** Serialized local copy saved to `tb_burden_2024_engineered.csv`.
* **Reproducibility & Security:** Fully decoupled database credentials using `.env` environment variables and zero-error pre-export guardrails.

---

### Next Phase: Exploratory Data Analysis (EDA) & Portfolio Visualization

With relational data persistence established, the analytical focus now shifts to:
1. **Demographic Analysis:** Stratifying TB incidence across age brackets and gender groups (`Female` vs. `Male`).
2. **Comorbidity & Risk Profile Analysis:** Quantifying burden drivers across reporting high-burden countries (`hiv`, `und`, `smk`, `alc`, `dia`).
3. **Geographic & Economic Disparity Profiling:** Pareto concentration across WHO regions and World Bank income classifications.